# 04 - Model Değerlendirme, Çapraz Doğrulama ve Veri Sızıntısı

> **Kapsam ve Hedef**: Sınıflandırma metrikleri (Confusion Matrix, Precision-Recall, F1, ROC-AUC), k-katlamalı çapraz doğrulama stratejileri ve Veri Sızıntısını (Data Leakage) önleme disiplini.

Bu notebook, **Keystone Mühendislik Prensipleri** doğrultusunda temiz kod, tip güvenliği ve veri bilimi odaklı bir yaklaşımla hazırlanmıştır.

---


### 1. Dengesiz Verilerde Doğruluk Tuzağı (Accuracy Paradox)
Sınıfların %95'i negatif olduğunda, her şeye 'Negatif' diyen bir model %95 Accuracy verir ancak hiçbir pozitif vakayı bulamaz. Bu yüzden Precision, Recall ve F1-Score kullanılır.


In [ ]:
import numpy as np

# Dengesiz Sınıf Dağılımı Simülasyonu (Örn: Sahtekarlık/Fraud Tespiti)
y_true = np.array([0]*95 + [1]*5)
y_pred_naive = np.array([0]*100)  # Her şeye sıfır diyen aptal model

acc = np.mean(y_true == y_pred_naive)
print(f"Aldatıcı Doğruluk (Accuracy): {acc:.2%}")

# TP, FP, FN, TN Hesaplama
tp = np.sum((y_true == 1) & (y_pred_naive == 1))
fp = np.sum((y_true == 0) & (y_pred_naive == 1))
fn = np.sum((y_true == 1) & (y_pred_naive == 0))
tn = np.sum((y_true == 0) & (y_pred_naive == 0))

recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Gerçek Pozitifleri Yakalama Oranı (Recall/Sensitivity): {recall:.2%}")
print("-> Model hiçbir sahtekarlığı yakalayamamıştır!")


### 2. Keystone İlkesi: Veri Sızıntısını (Data Leakage) Önleme
Ön işleme (scaling, imputation, encoding) adımları kesinlikle yalnızca Train kümesi üzerinde öğrenilmeli (`fit`), ardından Test kümesine uygulanmalıdır (`transform`).


In [ ]:
def safe_train_test_split(data_x, data_y, test_ratio=0.2, seed=42):
    np.random.seed(seed)
    indices = np.random.permutation(len(data_x))
    test_size = int(len(data_x) * test_ratio)
    test_idx, train_idx = indices[:test_size], indices[test_size:]
    return data_x[train_idx], data_x[test_idx], data_y[train_idx], data_y[test_idx]

X_dummy = np.arange(100).reshape(50, 2)
y_dummy = np.random.randint(0, 2, 50)

X_train, X_test, y_train, y_test = safe_train_test_split(X_dummy, y_dummy)
print(f"Eğitim Kümesi: {X_train.shape}, Test Kümesi: {X_test.shape}")
